In [1]:
# ===============================
# 📦 IMPORTS
# ===============================
import os
import pandas as pd
import numpy as np
import joblib

import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_squared_error, r2_score, mean_squared_log_error

In [ ]:
import mlflow

mlflow.set_tracking_uri("file:f:/streamlit_session/guvi_projects/smart_premium/mlruns")

mlflow.set_experiment("Insurance_Premium_Project")

print("✅ CLEAN FILE-BASED MLflow ACTIVE")

f:\streamlit_session\env\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
Traceback (most recent call last):
  File "f:\streamlit_session\env\Lib\site-packages\mlflow\store\tracking\file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
  File "f:\streamlit_session\env\Lib\site-packages\mlflow\store\tracking\file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
  File "f:\streamlit_session\env\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1670, in _read_yaml
    return _read_helper(root, 

✅ Using FILE-BASED MLflow ONLY


In [3]:

# ===============================
# 📊 LOAD DATA
# ===============================
df = pd.read_csv("data/processed/train_cleaned.csv")

df["log_premium"] = np.log1p(df["Premium Amount"])

X = df.drop(columns=["Premium Amount", "log_premium"], errors="ignore")
y = df["log_premium"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



In [4]:
# ===============================
# 🔧 PREPROCESSING (FIXED)
# ===============================
num_cols = X.select_dtypes(include="number").columns
cat_cols = X.select_dtypes(exclude="number").columns

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])


In [5]:
# ===============================
# 🤖 MODELS
# ===============================
models = {
    "LinearRegression": LinearRegression(),

    "RandomForest": RandomForestRegressor(
        n_estimators=150,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=150,
        learning_rate=0.08,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1,
        n_jobs=-1,
        random_state=42
    )
}



In [15]:
# ===============================
# 🏋️ TRAINING LOOP
# ===============================
results = {}

best_model = None
best_score = float("inf")
best_model_name = None

for name, model in models.items():

    with mlflow.start_run(run_name=name):

        print(f"\n🚀 Training {name}...")

        # Pipeline
        pipe = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        # Train
        pipe.fit(X_train, y_train)

        # Predict
        y_pred = pipe.predict(X_test)

        # Metrics
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)

        # SAFE RMSLE (avoids log issues)
        try:
            rmsle = np.sqrt(mean_squared_log_error(
                np.expm1(y_test),
                np.expm1(y_pred)
            ))
        except:
            rmsle = np.nan

        # ===============================
        # LOG TO MLFLOW
        # ===============================
        mlflow.log_param("model_name", name)
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2", r2)
        mlflow.log_metric("rmsle", rmsle)

        mlflow.sklearn.log_model(pipe, "model")

        print(f"✅ {name} logged")
        print(f"{name} → RMSE: {rmse:.4f}, R2: {r2:.4f}, RMSLE: {rmsle:.4f}")

        # Save results
        results[name] = (rmse, r2, rmsle)

        # Best model selection
        if rmsle < best_score:
            best_score = rmsle
            best_model = pipe
            best_model_name = name





🚀 Training LinearRegression...


2026/05/07 15:04:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 15:04:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ LinearRegression logged
LinearRegression → RMSE: 1.0896, R2: 0.0124, RMSLE: 1.0896

🚀 Training RandomForest...


2026/05/07 15:05:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 15:05:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ RandomForest logged
RandomForest → RMSE: 1.0640, R2: 0.0583, RMSLE: 1.0640

🚀 Training XGBoost...


2026/05/07 15:05:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/07 15:05:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ XGBoost logged
XGBoost → RMSE: 1.0545, R2: 0.0749, RMSLE: 1.0545


In [16]:
# ===============================
# 💾 SAVE BEST MODEL
# ===============================
os.makedirs("model", exist_ok=True)

joblib.dump(best_model, "model/best_model.pkl")

print("\n✅ Best model saved successfully!")
print(f"🏆 Best Model: {best_model_name}")


# ===============================
# 📊 FINAL COMPARISON
# ===============================
print("\n📊 Final Model Comparison:\n")

for name, scores in results.items():
    print(
        f"{name}: "
        f"RMSE={scores[0]:.4f}, "
        f"R2={scores[1]:.4f}, "
        f"RMSLE={scores[2]:.4f}"
    )


✅ Best model saved successfully!
🏆 Best Model: XGBoost

📊 Final Model Comparison:

LinearRegression: RMSE=1.0896, R2=0.0124, RMSLE=1.0896
RandomForest: RMSE=1.0640, R2=0.0583, RMSLE=1.0640
XGBoost: RMSE=1.0545, R2=0.0749, RMSLE=1.0545


In [13]:
import mlflow

print("Tracking URI:", mlflow.get_tracking_uri())
print("\nExperiments:")
print(mlflow.search_experiments())

print("\nRuns:")
print(mlflow.search_runs())

Tracking URI: sqlite:///f:/streamlit_session/guvi_projects/smart_premium/mlflow.db

Experiments:
[<Experiment: artifact_location='file:f:/streamlit_session/guvi_projects/smart_premium/mlruns/1', creation_time=1778145540353, experiment_id='1', last_update_time=1778145540353, lifecycle_stage='active', name='Insurance_Premium_Project', tags={}, trace_location=None, workspace='default'>, <Experiment: artifact_location='file:f:/streamlit_session/guvi_projects/smart_premium/mlruns/0', creation_time=1778145540348, experiment_id='0', last_update_time=1778145540348, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

Runs:
                             run_id experiment_id    status  \
0  1ef4c27638ec421aad7e31c044963510             1  FINISHED   
1  e708c5bdd31542c3884932410d6de1f6             1  FINISHED   
2  9a364a8f147f4e2cbe77fb5758972660             1  FINISHED   
3  0bbff78d507f4d39840f5275c34fa383             1  FINISHED   
4  e829881cb3764bd5a

In [17]:
import mlflow

print("Tracking URI:", mlflow.get_tracking_uri())


Tracking URI: file:f:/streamlit_session/guvi_projects/smart_premium/mlruns
